# 01 · Nighttime light measures for every survey region

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fundidayyan-afk/Measuring-Electrification-from-Space/blob/main/notebooks/01_light_measures.ipynb)

**What this notebook does.** For every DHS region (all eight surveys, every boundary level) and every year of VIIRS data from 2013 onward, it computes:

| Column | Meaning |
|---|---|
| `mean_rad` | Average radiance across the region (nW/cm²/sr), from the `average_masked` band with masked pixels counted as dark |
| `log_rad` | log(1 + `mean_rad`), the brightness measure used in the models |
| `lit_land_share` | Share of the region's land where radiance is above the background threshold |
| `lit_pop_share` | Share of the region's **people** who live in a lit pixel (WorldPop) |
| `lit_land_share_eog` / `lit_pop_share_eog` | The same two shares using the Earth Observation Group's own mask instead of our threshold (robustness check) |
| `pop` | Population in the region (WorldPop) |
| `cf_cvg_mean`, `low_cvg_share` | Average cloud-free nights per pixel, and the share of pixels with fewer than 10 (data quality) |

It also measures the **background threshold** from large protected areas with no settlement, as the proposal describes, and the same measures for each **whole country** (for Model 1).

**How to run it:** click **Runtime → Run all**. A Google sign-in window opens once; choose the account that owns the Earth Engine project. The full run takes roughly 10 to 30 minutes. At the end, two CSV files download to your computer.

## 1. Connect to Earth Engine

In [ ]:
PROJECT = "electrification-from-space"   # your Earth Engine Cloud project ID

import ee
ee.Authenticate()
ee.Initialize(project=PROJECT)
print("Connected to Earth Engine project:", PROJECT)

## 2. Get the region boundaries from the GitHub repo

In [ ]:
!rm -rf repo && git clone -q --depth 1 https://github.com/fundidayyan-afk/Measuring-Electrification-from-Space.git repo
!pip -q install geopandas

import glob, json, os, time
import geopandas as gpd
import pandas as pd

files = sorted(glob.glob("repo/data/boundaries/*.geojson"))
print(len(files), "boundary files found")

In [ ]:
# Simplify outlines slightly (0.002 degrees is about 220 m, finer than a 460 m VIIRS pixel)
# so the shapes are small enough to send to Earth Engine.
TOL = 0.002
KEEP = ["SVYID", "CNTRYNAMEE", "ISO", "REG_ID", "REGCODE", "REGNAME", "DHSREGEN", "LEVELRNK"]

layers = {}
for f in files:
    name = os.path.basename(f).replace(".geojson", "")            # e.g. KE2022DHS_sdr_subnational_boundaries
    survey, layer = name.split("_", 1)
    g = gpd.read_file(f).to_crs(4326)
    g["geometry"] = g.geometry.simplify(TOL, preserve_topology=True)
    g["survey"], g["layer"] = survey, layer
    g["uid"] = survey + "|" + layer + "|" + g["REG_ID"].astype(str)
    layers[name] = g[[c for c in KEEP if c in g] + ["survey", "layer", "uid", "geometry"]]

# Whole-country outlines, built by merging the regions of each country's latest survey
NATIONAL_FROM = {
    "KEN": "KE2022DHS_sdr_subnational_boundaries",
    "TZA": "TZ2022DHS_sdr_subnational_boundaries2",
    "UGA": "UG2024MIS_sdr_subnational_boundaries",
    "RWA": "RW2019DHS_sdr_subnational_boundaries",
}
nat = []
for iso3, src in NATIONAL_FROM.items():
    d = layers[src].dissolve()[["geometry"]]
    d["geometry"] = d.geometry.simplify(TOL * 2, preserve_topology=True)
    d["uid"], d["survey"], d["layer"], d["iso3"] = "national|" + iso3, "national", "national", iso3
    nat.append(d)
national = gpd.GeoDataFrame(pd.concat(nat, ignore_index=True), crs=4326)

def to_ee(gdf):
    """Convert a GeoDataFrame to an Earth Engine FeatureCollection."""
    feats = []
    for _, row in gdf.iterrows():
        props = {k: (None if pd.isna(v) else (v.item() if hasattr(v, "item") else v))
                 for k, v in row.drop("geometry").items()}
        geom = ee.Geometry(row.geometry.__geo_interface__, "EPSG:4326", False)
        feats.append(ee.Feature(geom, props))
    return ee.FeatureCollection(feats)

for k, g in layers.items():
    print(f"{k:45s} {len(g):3d} regions")

## 3. Light and population data

In [ ]:
VIIRS = ee.ImageCollection("NOAA/VIIRS/DNB/ANNUAL_V22")
POP = ee.ImageCollection("WorldPop/GP/100m/R2025A/v1/pop")
ISO3 = ["KEN", "TZA", "UGA", "RWA"]
POP_FIRST_YEAR = 2015   # WorldPop R2025A starts in 2015; earlier light years use the 2015 grid

# Which VIIRS years actually exist? (2012 is a partial-year composite and is excluded)
viirs_years = sorted(set(
    int(y) for y in VIIRS.aggregate_array("system:time_start")
    .map(lambda t: ee.Date(t).get("year")).getInfo()
))
YEARS = [y for y in viirs_years if y >= 2013]
print("VIIRS years used:", YEARS)

viirs_proj = VIIRS.first().select("average").projection()

def viirs(year):
    return VIIRS.filter(ee.Filter.calendarRange(year, year, "year")).first()

def pop_on_viirs_grid(year):
    """WorldPop people per 100 m cell, summed up to the ~460 m VIIRS grid."""
    y = max(year, POP_FIRST_YEAR)
    img = (POP.filter(ee.Filter.eq("year", y))
              .filter(ee.Filter.inList("country", ISO3))
              .mosaic().select("population").unmask(0))
    return (img.reduceResolution(ee.Reducer.sum(), maxPixels=1024)   # area-weighted sum
               .reproject(viirs_proj)
               .rename("pop"))

## 4. Background threshold

Pixels inside large, strictly protected areas (IUCN categories Ia, Ib and II, over 1,000 km²) where WorldPop shows **no people** should be dark. Whatever radiance VIIRS records there is noise. We take the 99th percentile of that noise in each year and use the **highest yearly value** as the threshold for "lit". This rule was fixed in `docs/analysis_plan.md` before any model was run.

In [ ]:
wdpa = (ee.FeatureCollection("WCMC/WDPA/current/polygons")
        .filter(ee.Filter.inList("ISO3", ISO3))
        .filter(ee.Filter.inList("IUCN_CAT", ["Ia", "Ib", "II"]))
        .filter(ee.Filter.eq("MARINE", "0"))
        .filter(ee.Filter.gt("GIS_AREA", 1000)))
print("Protected areas used:", wdpa.size().getInfo())
assert wdpa.size().getInfo() > 0, "No protected areas matched; check the WDPA filters"
pa_geom = wdpa.geometry()

rows = []
for y in YEARS:
    unsettled = pop_on_viirs_grid(y).lte(0)
    noise = viirs(y).select("average").updateMask(unsettled)
    stats = noise.reduceRegion(
        reducer=ee.Reducer.percentile([50, 95, 99]).combine(ee.Reducer.count(), sharedInputs=True),
        geometry=pa_geom, scale=463.83, maxPixels=1e10, tileScale=4,
    ).getInfo()
    stats["year"] = y
    rows.append(stats)
    print(y, {k: round(v, 3) if isinstance(v, float) else v for k, v in stats.items()})

threshold_table = pd.DataFrame(rows)
THRESHOLD = float(threshold_table["average_p99"].max())
print(f"\nLit threshold = {THRESHOLD:.3f} nW/cm²/sr")

## 5. Measures for every region and year

In [ ]:
def year_bands(y):
    v = viirs(y)
    rad = v.select("average_masked").unmask(0)               # EOG-masked radiance; masked = dark
    avg = v.select("average")
    pop = pop_on_viirs_grid(y)
    lit = avg.gt(THRESHOLD)                                  # our threshold
    lit_eog = rad.gt(0)                                      # EOG's own mask
    cvg = v.select("cf_cvg")
    return ee.Image.cat([
        rad.rename(f"rad_{y}"),
        lit.rename(f"lit_{y}"),
        lit_eog.rename(f"liteog_{y}"),
        pop.rename(f"pop_{y}"),
        pop.multiply(lit).rename(f"litpop_{y}"),
        pop.multiply(lit_eog).rename(f"litpopeog_{y}"),
        cvg.rename(f"cvg_{y}"),
        cvg.lt(10).rename(f"lowcvg_{y}"),
    ]).toFloat()

def measures(fc, label):
    """Pixel means over each region for every year; sums of people follow from means."""
    out = []
    for y in YEARS:   # one request per year keeps each request small
        res = year_bands(y).reduceRegions(
            collection=fc, reducer=ee.Reducer.mean(), scale=463.83,
            crs=viirs_proj, tileScale=4,
        ).getInfo()["features"]
        for f in res:
            p = f["properties"]
            pop_mean, litpop, litpopeog = p.get(f"pop_{y}"), p.get(f"litpop_{y}"), p.get(f"litpopeog_{y}")
            rec = {k: v for k, v in p.items() if not k.endswith(f"_{y}")}
            rec.update({
                "year": y,
                "mean_rad": p.get(f"rad_{y}"),
                "lit_land_share": p.get(f"lit_{y}"),
                "lit_land_share_eog": p.get(f"liteog_{y}"),
                "lit_pop_share": (litpop / pop_mean) if pop_mean else None,
                "lit_pop_share_eog": (litpopeog / pop_mean) if pop_mean else None,
                "pop_density_per_pixel": pop_mean,
                "cf_cvg_mean": p.get(f"cvg_{y}"),
                "low_cvg_share": p.get(f"lowcvg_{y}"),
            })
            out.append(rec)
        print(f"  {label} {y}: {len(res)} regions")
    return out

def run_with_retry(gdf, label, tries=3):
    for i in range(tries):
        try:
            return measures(to_ee(gdf), label)
        except Exception as e:
            print(f"  retry {i+1} for {label}: {e}")
            time.sleep(10)
    raise RuntimeError(f"{label} failed after {tries} tries")

all_rows = []
start = time.time()
for name, g in layers.items():
    print(name)
    all_rows += run_with_retry(g, name)
print("National outlines")
all_rows += run_with_retry(national, "national")
print(f"Done in {(time.time() - start) / 60:.1f} minutes")

## 6. Tidy up, check, and download

In [ ]:
import numpy as np

df = pd.DataFrame(all_rows)
df["log_rad"] = np.log1p(df["mean_rad"])
ISO_FROM_SURVEY = {"KE": "KEN", "TZ": "TZA", "UG": "UGA", "RW": "RWA"}
df["iso3"] = df.get("iso3", pd.Series(index=df.index, dtype=object)).fillna(df["survey"].str[:2].map(ISO_FROM_SURVEY))
df["threshold"] = THRESHOLD
df["viirs_version"] = "NOAA/VIIRS/DNB/ANNUAL_V22"
df["pop_version"] = "WorldPop/GP/100m/R2025A/v1/pop"
df["run_date"] = pd.Timestamp.today().date().isoformat()

first = ["uid", "survey", "layer", "iso3", "SVYID", "CNTRYNAMEE", "REG_ID", "REGCODE", "REGNAME", "DHSREGEN",
         "LEVELRNK", "year", "mean_rad", "log_rad", "lit_land_share", "lit_pop_share",
         "lit_land_share_eog", "lit_pop_share_eog", "pop_density_per_pixel", "cf_cvg_mean", "low_cvg_share"]
df = df[[c for c in first if c in df] + [c for c in df if c not in first]]
df = df.sort_values(["survey", "layer", "REGNAME", "year"], na_position="last")

# Basic checks: shares between 0 and 1, no missing core values
for c in ["lit_land_share", "lit_pop_share", "lit_land_share_eog", "lit_pop_share_eog"]:
    bad = ((df[c] < 0) | (df[c] > 1)).sum()
    print(f"{c:22s} missing={df[c].isna().sum():4d}  out of range={bad}")
print(len(df), "rows")

df.to_csv("light_measures.csv", index=False)
threshold_table.to_csv("light_threshold.csv", index=False)
display(df[df.layer == "national"].pivot(index="year", columns="iso3", values="lit_pop_share").round(3))

In [ ]:
from google.colab import files
files.download("light_measures.csv")
files.download("light_threshold.csv")

**Last step:** send both downloaded files (`light_measures.csv` and `light_threshold.csv`) to Claude in the chat, or upload them to `data/light/` in the GitHub repo with **Add file → Upload files**.